# 02 RQ2: longshot bias and liquidity

Secondary test of RQ2 (`docs/hypotheses.md` Section 4), liquidity as defined in
Section 3 and clarified in D2.

- **Longshot:** in each 1-day snapshot, the cheaper side when it is priced below
  $0.10 (side-agnostic, T6). Its outcome is whether that side won.
- **H2a:** mean(won − price) over longshot snapshots is negative. One-sample test on
  cluster means, two-sided.
- **H2b:** in an OLS regression of (won − price) on log liquidity with format fixed
  effects and cluster-robust SEs, the liquidity coefficient is positive. Two-sided.
- α = 0.05 per hypothesis, Holm across H2a and H2b. Other horizons are descriptive.

**Status: gated.** D7 states that the application of Section 5 to RQ2 is fixed in a
later deviation entry *before RQ2 is analysed*, and Section 4 leaves the estimation
details below open. Until that entry is committed, `RUN_CONFIRMATORY = False`: the
outcome column is not loaded, and only the planted control on simulated outcomes runs.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.multitest import multipletests

RUN_CONFIRMATORY = False   # flip only after the RQ2 deviation entry is committed

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

SEED = 20261112          # D7.5, reused
B = 1000                 # bootstrap resamples (Section 4)
LONGSHOT = 0.10          # Section 4
PRIMARY_H = "1d"         # Section 4

pd.set_option("display.width", 200)

cols = ["market_id", "horizon", "snap_price", "is_primary", "liquidity_usd",
        "format", "cluster_id"] + (["token1_won"] if RUN_CONFIRMATORY else [])
frame = pd.read_parquet(DATA / "mart_analysis.parquet", columns=cols)
frame = frame[frame["is_primary"]].copy()

# Cheaper side (T6): token1 when p < 0.5, else token2 at price 1 - p.
frame["ls_is_token1"] = frame["snap_price"] < 0.5
frame["ls_price"] = np.where(frame["ls_is_token1"], frame["snap_price"], 1 - frame["snap_price"])
frame["log_liq"] = np.log1p(frame["liquidity_usd"])       # P3
frame["template"] = (frame["format"] == "template").astype(float)
if RUN_CONFIRMATORY:
    frame["won"] = np.where(frame["ls_is_token1"], frame["token1_won"], ~frame["token1_won"]).astype(float)

ls = frame[frame["ls_price"] < LONGSHOT]
ls1d = ls[ls["horizon"] == PRIMARY_H].reset_index(drop=True)
ls.groupby(["horizon", "format"]).agg(
    snapshots=("ls_price", "size"), clusters=("cluster_id", "nunique"),
    zero_liquidity=("liquidity_usd", lambda s: int((s <= 0).sum())),
    median_price=("ls_price", "median"),
)

snapshots  clusters  zero_liquidity  median_price
horizon format                                                     
1d      event         50466     10563            1707         0.020
        template      93954       529             418         0.010
1h      event        186184      9616             143         0.001
        template      75241       485              44         0.001
1w      event         30345      8315             584         0.016
        template       9253       266             919         0.020

## 1. Estimation details (proposed, pending a deviation entry)

Section 4 fixes the hypotheses; these choices are open and are proposed here, mirroring
D7 for RQ1. None depends on an outcome.

- **P1. H2a test.** Per cluster (Section 3 clusters), the mean of (won − price) over
  its 1-day longshot snapshots, both formats pooled. A one-sample t test of the
  unweighted cluster means against 0 (df = G − 1), two-sided, with a 95% t interval.
  The snapshot-level pooled mean is reported beside it, descriptive.
- **P2. H2b model.** OLS of (won − price) on log liquidity and a template indicator
  (format fixed effect), CR1 cluster-robust SEs on the Section 3 clusters, z
  reference distribution (as D7.1), two-sided, 95% Wald CI.
- **P3. Zero liquidity.** Liquidity is log(1 + USD), so a snapshot that is the
  market's first trade (liquidity 0) stays in the sample at 0.
- **P4. Multiplicity.** Holm across H2a and H2b at family-wise α = 0.05.
- **P5. Descriptives.** For every horizon × format: snapshot count, clusters, mean
  (won − price) with a 1,000-resample cluster-bootstrap percentile CI (seed as D7.5).
  At 1 day: the same by liquidity quintile.
- **P6. Sensitivities.** The Section 5 list for RQ2 is to be fixed in the deviation
  entry; no sensitivity cell exists yet.

In [2]:
def h2a(d, y):
    """P1: one-sample t test on unweighted cluster means of (y - price)."""
    resid = y - d["ls_price"].to_numpy()
    g = pd.factorize(d["cluster_id"])[0]
    m = np.bincount(g, resid) / np.bincount(g)
    G = len(m)
    se = m.std(ddof=1) / np.sqrt(G)
    t = m.mean() / se
    q = stats.t.ppf(0.975, G - 1)
    return dict(test="H2a", estimate=m.mean(), se=se, ci_low=m.mean() - q * se,
                ci_high=m.mean() + q * se, p_value=2 * stats.t.sf(abs(t), G - 1),
                clusters=G, n=len(d), pooled_mean=resid.mean())


def h2b(d, y):
    """P2: OLS of (y - price) on log liquidity + format FE, CR1 cluster SEs, z."""
    X = np.column_stack([np.ones(len(d)), d["log_liq"].to_numpy(), d["template"].to_numpy()])
    g = pd.factorize(d["cluster_id"])[0]
    res = sm.OLS(y - d["ls_price"].to_numpy(), X).fit(cov_type="cluster", cov_kwds={"groups": g})
    b, se = res.params[1], res.bse[1]
    return dict(test="H2b", estimate=b, se=se, ci_low=b - 1.959964 * se,
                ci_high=b + 1.959964 * se, p_value=res.pvalues[1],
                clusters=g.max() + 1, n=len(d), pooled_mean=np.nan)


def ols_truth(d, q):
    """Population H2b coefficient: OLS of the true (q - price) on the same design."""
    X = np.column_stack([np.ones(len(d)), d["log_liq"].to_numpy(), d["template"].to_numpy()])
    return np.linalg.lstsq(X, q - d["ls_price"].to_numpy(), rcond=None)[0][1]


def h2a_truth(d, q):
    g = pd.factorize(d["cluster_id"])[0]
    return (np.bincount(g, q - d["ls_price"].to_numpy()) / np.bincount(g)).mean()

## 2. Planted control (simulated outcomes only)

The real prices, liquidity, formats and clusters of the 1-day longshot frame are kept;
only the outcome is simulated, so the check exercises the exact sample and cluster
structure the test will see.

- **Planted:** true win probability q = price × (1 − k), with
  k = clip(0.30 − 0.05 × (log liquidity − median), 0, 1). Longshots are overpriced by
  30% at median liquidity, less so in deeper markets: H2a's truth is negative and
  H2b's truth is positive.
- **Null:** q = price, so both truths are 0.

Each scenario is drawn 50 times. The truths are computed from q itself: H2a's is the
mean of cluster means of (q − price), H2b's is the OLS coefficient of (q − price) on
the same design (E[won | X] = q, so OLS on simulated outcomes is unbiased for it).
Pass: 95% CIs cover the truth in roughly 95% of draws in both scenarios, the planted
effects are rejected in most draws, and the null rejection rate is near 5%.

In [3]:
rng = np.random.default_rng(SEED)
R = 50
k = np.clip(0.30 - 0.05 * (ls1d["log_liq"] - ls1d["log_liq"].median()), 0, 1).to_numpy()
scenarios = {"planted": ls1d["ls_price"].to_numpy() * (1 - k), "null": ls1d["ls_price"].to_numpy()}

rows = []
for name, q in scenarios.items():
    truth = {"H2a": h2a_truth(ls1d, q), "H2b": ols_truth(ls1d, q)}
    for r in range(R):
        y = rng.binomial(1, q).astype(float)
        for fit in (h2a(ls1d, y), h2b(ls1d, y)):
            rows.append({"scenario": name, "rep": r, "truth": truth[fit["test"]], **fit})
pc = pd.DataFrame(rows)
pc["covers"] = (pc["ci_low"] <= pc["truth"]) & (pc["truth"] <= pc["ci_high"])
pc["rejects_0"] = pc["p_value"] < 0.05
summary = pc.groupby(["scenario", "test"]).agg(
    truth=("truth", "first"), mean_estimate=("estimate", "mean"), mean_se=("se", "mean"),
    sd_estimate=("estimate", "std"), coverage=("covers", "mean"), reject_rate=("rejects_0", "mean"),
)
summary.to_csv(RESULTS / "rq2_planted_control.csv")
summary

truth  mean_estimate   mean_se  sd_estimate  coverage  reject_rate
scenario test                                                                       
null     H2a   0.000000       0.000141  0.001195     0.001244      0.90         0.10
         H2b   0.000000      -0.000021  0.000137     0.000130      0.98         0.02
planted  H2a  -0.006426      -0.006530  0.001045     0.001061      0.96         1.00
         H2b   0.001626       0.001648  0.000133     0.000105      0.98         1.00

In [4]:
checks = {
    "coverage >= 0.88 in every scenario x test": bool((summary["coverage"] >= 0.88).all()),
    "planted effects rejected in >= 80% of draws": bool((summary.loc["planted", "reject_rate"] >= 0.80).all()),
    "planted signs right (H2a < 0, H2b > 0)": bool(summary.loc[("planted", "H2a"), "mean_estimate"] < 0
                                                   and summary.loc[("planted", "H2b"), "mean_estimate"] > 0),
    "null rejection rate <= 0.12": bool((summary.loc["null", "reject_rate"] <= 0.12).all()),
}
for name, ok in checks.items():
    print(("PASS " if ok else "FAIL ") + name)
PLANTED_CONTROL_PASSED = all(checks.values())

PASS coverage >= 0.88 in every scenario x test
PASS planted effects rejected in >= 80% of draws
PASS planted signs right (H2a < 0, H2b > 0)
PASS null rejection rate <= 0.12


## 3. Confirmatory tests (gated)

Runs only with `RUN_CONFIRMATORY = True` and a passing planted control. H2a and H2b at
the 1-day horizon, Holm across the two.

In [5]:
if RUN_CONFIRMATORY and PLANTED_CONTROL_PASSED:
    y = ls1d["won"].to_numpy()
    tests = pd.DataFrame([h2a(ls1d, y), h2b(ls1d, y)])
    tests["p_holm"] = multipletests(tests["p_value"], method="holm")[1]
    tests["significant_holm"] = tests["p_holm"] < 0.05
    tests.to_csv(RESULTS / "rq2_tests.csv", index=False)
    display(tests.round(5))
else:
    print("confirmatory tests skipped (gate closed)")

confirmatory tests skipped (gate closed)


## 4. Descriptives (gated)

P5: mean(won − price) per horizon × format, and by liquidity quintile at 1 day, with
cluster-bootstrap 95% CIs.

In [6]:
def boot_mean(d, B=B):
    resid = d["won"].to_numpy() - d["ls_price"].to_numpy()
    g, _ = pd.factorize(d["cluster_id"])
    C = g.max() + 1
    W = rng_desc.multinomial(C, np.full(C, 1 / C), size=B).astype(float)
    b = (W @ np.bincount(g, resid, C)) / (W @ np.bincount(g, minlength=C))
    lo, hi = np.percentile(b, [2.5, 97.5])
    return dict(n=len(d), clusters=C, mean_resid=resid.mean(), lo=lo, hi=hi)


if RUN_CONFIRMATORY:
    rng_desc = np.random.default_rng(SEED)
    desc = pd.DataFrame([{"horizon": h, "format": f, **boot_mean(d)}
                         for (h, f), d in ls.groupby(["horizon", "format"])])
    desc.to_csv(RESULTS / "rq2_descriptives.csv", index=False)
    q = ls1d.assign(liq_quintile=pd.qcut(ls1d["log_liq"], 5, labels=False, duplicates="drop") + 1)
    quint = pd.DataFrame([{"liq_quintile": k, "median_liq_usd": d["liquidity_usd"].median(), **boot_mean(d)}
                          for k, d in q.groupby("liq_quintile")])
    quint.to_csv(RESULTS / "rq2_liquidity_quintiles.csv", index=False)
    display(desc.round(5), quint.round(5))
else:
    print("descriptives skipped (gate closed)")

descriptives skipped (gate closed)
